In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
import json, os, re, time, subprocess, ast
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

INPUT_DIR = '/kaggle/input/datasets/'
OUTPUT_DIR = '/kaggle/working'

TEST_PATH = f'{INPUT_DIR}/test_loop_filtered_verified.jsonl'
TEST_CASES_DIR = f'{INPUT_DIR}/public_test_cases/codenet/public_test_cases'
PROMPTING_OUTPUT_PATH = f'{OUTPUT_DIR}/prompting_baseline_gpt54mini.jsonl'

SPEEDUP_THRESHOLD = 1.10
MAX_ATTEMPTS = 7
GPT_MODEL = "gpt-5.4-mini"

user_secrets = UserSecretsClient()
client = OpenAI(api_key=user_secrets.get_secret("OPENAI_API_KEY"))

test_samples = []
with open(TEST_PATH) as f:
    for line in f:
        test_samples.append(json.loads(line))
print(f'Loaded {len(test_samples)} test samples')

print('Test cases dir exists:', os.path.isdir(TEST_CASES_DIR))

In [ ]:
def call_gen_llm(prompt, max_new_tokens=800, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.15,
                max_completion_tokens=max_new_tokens,
            )
            content = response.choices[0].message.content
            return content.strip() if content else ''
        except Exception as e:
            err_str = str(e).lower()
            if 'rate_limit' in err_str or '429' in err_str:
                wait_time = 15 * (attempt + 1)
                print(f'   Rate limited, waiting {wait_time}s...')
                time.sleep(wait_time)
            else:
                print(f'   API error: {e}')
                return ''
    return ''

print(f'{GPT_MODEL} ready.')

In [ ]:
def extract_code_from_response(response):
    match = re.search(r"```(?:python)?\s*\n(.*?)```", response, re.DOTALL)
    if match:
        return match.group(1).strip()
    return response.strip()

print('Code extractor ready.')

In [ ]:
def load_test_cases(problem_id, test_cases_dir):
    problem_dir = os.path.join(test_cases_dir, problem_id)
    if not os.path.isdir(problem_dir):
        return []
    cases = []
    i = 0
    while True:
        input_path = os.path.join(problem_dir, f'input.{i}.txt')
        output_path = os.path.join(problem_dir, f'output.{i}.txt')
        if not os.path.exists(input_path):
            break
        with open(input_path) as f:
            stdin_data = f.read()
        with open(output_path) as f:
            expected_output = f.read().strip()
        cases.append({'stdin': stdin_data, 'expected': expected_output})
        i += 1
    return cases

def check_compiles(code):
    try:
        compile(code, '<repair>', 'exec')
        return True, None
    except SyntaxError as e:
        return False, str(e)

def run_with_timeout(code, stdin_data, timeout_sec=5):
    start = time.time()
    try:
        result = subprocess.run(['python3', '-c', code], input=stdin_data,
                                 capture_output=True, text=True, timeout=timeout_sec)
        elapsed = time.time() - start
        return result.stdout.strip(), result.returncode == 0, elapsed
    except subprocess.TimeoutExpired:
        return None, False, timeout_sec

def check_functional_correctness(code, test_cases):
    for case in test_cases:
        output, success, _ = run_with_timeout(code, case['stdin'])
        if not success or output != case['expected']:
            return False, {**case, 'actual_output': output if output is not None else '(crashed or timed out)'}
    return True, None

def measure_runtime(code, test_cases, repeats=3):
    total_time = 0
    for _ in range(repeats):
        for case in test_cases:
            _, success, elapsed = run_with_timeout(code, case['stdin'])
            if not success:
                return None
            total_time += elapsed
    return total_time / repeats

def validate_repair(slow_code, repaired_code, problem_id, test_cases_dir, speedup_threshold=SPEEDUP_THRESHOLD):
    test_cases = load_test_cases(problem_id, test_cases_dir)
    if not test_cases:
        return {'verdict': 'RETRY', 'reason': f'No test cases found for {problem_id}'}
    compiles, syntax_error = check_compiles(repaired_code)
    if not compiles:
        return {'verdict': 'RETRY', 'reason': 'compile_error', 'detail': syntax_error}
    correct, failing_case = check_functional_correctness(repaired_code, test_cases)
    if not correct:
        return {'verdict': 'RETRY', 'reason': 'test_failure', 'detail': failing_case}
    slow_runtime = measure_runtime(slow_code, test_cases)
    repaired_runtime = measure_runtime(repaired_code, test_cases)
    if slow_runtime is None or repaired_runtime is None or repaired_runtime <= 0:
        return {'verdict': 'RETRY', 'reason': 'runtime_measurement_failed'}
    speedup = slow_runtime / repaired_runtime
    if speedup >= speedup_threshold:
        return {'verdict': 'ACCEPT', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}
    else:
        return {'verdict': 'RETRY', 'reason': 'insufficient_speedup', 'speedup': round(speedup, 3),
                'slow_runtime': round(slow_runtime, 4), 'repaired_runtime': round(repaired_runtime, 4)}

def load_completed_problem_ids(output_path):
    completed = set()
    if os.path.exists(output_path):
        with open(output_path) as f:
            for line in f:
                try:
                    completed.add(json.loads(line)['problem_id'])
                except Exception:
                    continue
    return completed

print('Validation + checkpoint helpers ready.')

In [ ]:
def build_prompting_baseline_prompt(slow_code):
    return f"""You are an expert Python engineer specializing in performance optimization.

Optimize the following Python code to run faster while preserving its exact behavior.

Slow code:
{slow_code}

Return ONLY the optimized Python code. No markdown. No explanations."""


def build_prompting_feedback_prompt(slow_code, previous_code, validation_result):
    reason = validation_result.get('reason')
    if reason == 'compile_error':
        feedback = f"Your previous attempt had a syntax error: {validation_result.get('detail')}"
    elif reason == 'test_failure':
        case = validation_result.get('detail', {})
        feedback = (
            f"Your previous attempt failed a test case.\n"
            f"Input:\n{case.get('stdin', '')}\n"
            f"Expected output:\n{case.get('expected', '')}\n"
            f"Your code's ACTUAL output:\n{case.get('actual_output', '')}\n"
            f"Fix the logic error."
        )
    elif reason == 'insufficient_speedup':
        feedback = (
            f"Your previous attempt was correct but NOT fast enough "
            f"({validation_result.get('speedup')}x, need >= {SPEEDUP_THRESHOLD}x). "
            f"Try a genuinely more efficient approach."
        )
    else:
        feedback = "Your previous attempt did not pass validation. Try a different approach."

    base_prompt = build_prompting_baseline_prompt(slow_code)
    return f"""{base_prompt}

Your previous attempt:
{previous_code}

Feedback on that attempt:
{feedback}

Generate a NEW, corrected version that addresses this feedback."""


def prompting_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts=MAX_ATTEMPTS):
    prompt = build_prompting_baseline_prompt(slow_code)
    current_code = extract_code_from_response(call_gen_llm(prompt, max_new_tokens=800))
    history = []

    for attempt in range(1, max_attempts + 1):
        result = validate_repair(slow_code, current_code, problem_id, test_cases_dir)
        result['attempt'] = attempt
        history.append(result)

        if result['verdict'] == 'ACCEPT':
            return {'final_verdict': 'ACCEPT', 'code': current_code, 'history': history}
        if attempt == max_attempts:
            return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}

        fb_prompt = build_prompting_feedback_prompt(slow_code, current_code, result)
        current_code = extract_code_from_response(call_gen_llm(fb_prompt, max_new_tokens=800))

    return {'final_verdict': 'RETRY_EXHAUSTED', 'code': current_code, 'history': history}


def run_prompting_baseline(test_samples, test_cases_dir, output_path, max_attempts=MAX_ATTEMPTS):
    completed = load_completed_problem_ids(output_path)
    print(f'Already completed: {len(completed)} / {len(test_samples)}')

    with open(output_path, 'a') as out_f:
        for i, sample in enumerate(test_samples):
            problem_id = sample.get('problem_id')
            slow_code = sample['input']
            if problem_id in completed:
                continue

            print(f'[{i+1}/{len(test_samples)}] {problem_id}...')
            start = time.time()
            try:
                loop_result = prompting_repair_loop(slow_code, problem_id, test_cases_dir, max_attempts)
                record = {
                    'problem_id': problem_id,
                    'final_verdict': loop_result['final_verdict'],
                    'num_attempts': len(loop_result['history']),
                    'history': loop_result['history'],
                    'final_code': loop_result['code'],
                    'elapsed_seconds': round(time.time() - start, 2),
                }
            except Exception as e:
                record = {'problem_id': problem_id, 'final_verdict': 'PIPELINE_ERROR', 'error': str(e)}

            out_f.write(json.dumps(record) + '\n')
            out_f.flush()

    print('Prompting baseline run complete.')

print('Prompting baseline logic ready.')

In [ ]:
import json, os, glob

OUT = "/kaggle/working/pf_gpt54mini_all302.jsonl"

# 1. find the uploaded file
found = glob.glob("/kaggle/input/datasets/bk9915110/pf-gpt/**/*.jsonl", recursive=True)
print("uploaded file:", found)
assert found, "No .jsonl file in that folder"

# 2. merge uploaded results with any new results
merged = {}
for path in found + [OUT]:
    if not os.path.exists(path):
        continue
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        i = r["test_index"]
        if i not in merged or merged[i].get("final_verdict") == "PIPELINE_ERROR":
            merged[i] = r

# 3. write the merged file
with open(OUT, "w") as f:
    for i in sorted(merged):
        f.write(json.dumps(merged[i]) + "\n")

ok = sum(r.get("final_verdict") != "PIPELINE_ERROR" for r in merged.values())
print("finished:", ok, "/ 302 | left:", 302 - ok)

In [ ]:
import os, json, time

OUT = "/kaggle/working/pf_gpt54mini_all302.jsonl"

# ---------- 1. check test set ----------
print("test samples:", len(test_samples))            # must be 302
assert len(test_samples) == 302

# ---------- 2. check output file ----------
done = set()
if os.path.exists(OUT):
    for line in open(OUT):
        r = json.loads(line)
        if "test_index" in r and r.get("final_verdict") != "PIPELINE_ERROR":
            done.add(r["test_index"])
print("already done:", len(done), "| left:", 302 - len(done))

# ---------- 3. run from first to last ----------
with open(OUT, "a") as f:
    for i in range(302):
        if i in done:
            continue
        s = test_samples[i]
        t = time.time()
        try:
            lr = prompting_repair_loop(s["input"], s["problem_id"], TEST_CASES_DIR, MAX_ATTEMPTS)
            rec = {"final_verdict": lr["final_verdict"], "num_attempts": len(lr["history"]),
                   "history": lr["history"], "final_code": lr["code"],
                   "elapsed_seconds": round(time.time() - t, 2)}
        except Exception as e:
            rec = {"final_verdict": "PIPELINE_ERROR", "error": str(e)}
        rec.update({"test_index": i, "problem_id": s["problem_id"]})
        f.write(json.dumps(rec) + "\n")
        f.flush()
        print(f"[{i+1}/302] {s['problem_id']} -> {rec['final_verdict']}"
              + (f"   ERROR: {rec['error']}" if "error" in rec else ""))

# ---------- 4. check result file and score ----------
latest = {}
for line in open(OUT):
    r = json.loads(line)
    latest[r["test_index"]] = r
rows = [latest[i] for i in sorted(latest)]

n   = len(rows)
acc = sum(r["final_verdict"] == "ACCEPT" for r in rows)
err = sum(r["final_verdict"] == "PIPELINE_ERROR" for r in rows)
c1  = sum(1 for r in rows if r.get("history") and
          (r["history"][0].get("verdict") == "ACCEPT" or
           r["history"][0].get("reason") == "insufficient_speedup"))
sp  = sorted(r["history"][-1]["speedup"] for r in rows
             if r["final_verdict"] == "ACCEPT" and "speedup" in r["history"][-1])

print(f"\nSamples: {n} (should be 302) | errors: {err}")
print(f"%OPT: {100*acc/n:.1f}%  |  Correct@1: {100*c1/n:.1f}%")
if sp:
    print(f"Mean speedup: {sum(sp)/len(sp):.3f}x  |  Median: {sp[len(sp)//2]:.3f}x")

In [ ]:
   from statsmodels.stats.contingency_tables import mcnemar
   # b = PyLoopFix yes, baseline no ; c = PyLoopFix no, baseline yes
   print(mcnemar([[0, b], [c, 0]], exact=True).pvalue)